<div style="background:linear-gradient(135deg,#123524 0%,#1f5a45 45%,#2B5A82 100%);color:#fff;padding:40px 36px;border-radius:16px;font-family:system-ui,-apple-system,'Segoe UI',sans-serif">
  <div style="font-size:12px;letter-spacing:.16em;opacity:.75;text-transform:uppercase">43008 Reinforcement Learning &middot; Assignment 3 &middot; Team LMF</div>
  <div style="font-size:46px;font-weight:800;line-height:1.05;margin:12px 0 6px">Experiment: PPO</div>
  <div style="font-size:19px;opacity:.92">Proximal Policy Optimisation on AgriFlight-RL, crop-row flight stabilisation</div>
  <div style="height:1px;background:rgba(255,255,255,.28);margin:24px 0 18px"></div>
  <div style="display:flex;flex-wrap:wrap;gap:36px">
    <div><div style="font-size:11px;letter-spacing:.12em;opacity:.7;text-transform:uppercase">Student</div><div style="font-size:20px;font-weight:600">Nelkit Isael Chavez Calona</div></div>
    <div><div style="font-size:11px;letter-spacing:.12em;opacity:.7;text-transform:uppercase">Student ID</div><div style="font-size:20px;font-weight:600">25609227</div></div>
    <div><div style="font-size:11px;letter-spacing:.12em;opacity:.7;text-transform:uppercase">Environment</div><div style="font-size:20px;font-weight:600">AgroDrone-v0 &middot; MuJoCo Skydio X2</div></div>
  </div>
</div>

### Contents

1. [Introduction](#intro): question, hypothesis, success criterion
2. [Setup](#setup): environment, config, seeds, shared helpers
3. [Theory](#theory): the algorithm in your own words
4. [Implementation](#implementation): the code
5. [Training](#training): run, curve, stability
6. [Evaluation](#evaluation): held-out seeds, baselines, difficulty levels
7. [Results](#results): table and plots
8. [Discussion](#discussion): what worked, what did not, limits
9. [Contribution and reproducibility](#contribution): who did what, versions, references

<div style="border-left:4px solid rgb(140,90,43);background:rgba(140,90,43,0.10);padding:10px 14px;border-radius:0 8px 8px 0;margin:6px 0"><b>Template status (delete this box before submitting).</b> <code>Run all</code> works today and produces the baseline numbers. Cells marked <i>Write here</i> or <code>TODO</code> are yours. Do not edit the cells labelled <i>Shared, identical in all three notebooks</i>: the comparison is only fair if they stay the same.</div>

<a id="intro"></a>

## 1. Introduction

<div style="border-left:4px solid rgb(147,105,28);background:rgba(147,105,28,0.10);padding:10px 14px;border-radius:0 8px 8px 0;margin:6px 0"><b>Write here.</b> In about 150 words:<ul><li>The research question this notebook answers (not "train a drone")</li><li>What PPO is, in two sentences</li><li>Your hypothesis. Part B expects PPO to be the most stable of the three. Say whether you agree, and what result would change your mind.</li><li><b>Your success criterion, written before you see results</b>: the highest difficulty level you expect to clear (L0 to L4) and roughly how many samples it needs</li></ul></div>

<a id="setup"></a>

## 2. Setup

**2.1 Bootstrap.** Finds or clones the repo, fetches the Skydio X2 model and installs dependencies. Works locally and in a fresh Colab runtime. Set `BRANCH` if the environment rework lives on another branch.

In [ ]:
import os, sys, subprocess, time
NOTEBOOK_T0 = time.time()

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Nelkit/agroflight-rl.git"   # private repo: use a token URL
BRANCH = "main"                                            # switch to the branch with the env rework once pushed
CLONE_DIR = "/content/agroflight-rl"

def run(cmd, cwd=None):
    subprocess.run(cmd, cwd=cwd, check=True)

def find_repo_root(start):
    p = os.path.abspath(start)
    while True:
        if os.path.isfile(os.path.join(p, "pyproject.toml")):
            return p
        parent = os.path.dirname(p)
        if parent == p:
            return None
        p = parent

REPO = find_repo_root(os.getcwd())
if REPO is None:
    if not os.path.isdir(CLONE_DIR):
        run(["git", "clone", "--branch", BRANCH, REPO_URL, CLONE_DIR])
    REPO = CLONE_DIR
os.chdir(REPO)

DRONE_XML = os.path.join(REPO, "mujoco_menagerie", "skydio_x2", "x2.xml")
if not os.path.isfile(DRONE_XML):
    if not os.path.isdir("mujoco_menagerie"):
        run(["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
             "https://github.com/google-deepmind/mujoco_menagerie.git", "mujoco_menagerie"])
    run(["git", "sparse-checkout", "set", "skydio_x2"], cwd="mujoco_menagerie")
os.environ["AGRO_DRONE_XML"] = DRONE_XML

try:
    import mujoco, gymnasium
except ImportError:
    run([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
sys.path.insert(0, os.path.join(REPO, "src"))

print("repo :", REPO)
print("model:", DRONE_XML)

**2.2 Experiment config and seeds.** Training seeds start at 0. Evaluation seeds start at 1000, so they are held out from training.

In [ ]:
import random
import numpy as np
import torch

EXPERIMENT = dict(
    algorithm="PPO",
    student_name="Nelkit Isael Chavez Calona",
    student_id="25609227",
    env_id="AgroDrone-v0",
    train_seed=0,
    eval_seeds=list(range(1000, 1020)),
    level=0,   # start at L0; move up only when it works
)
BACKUP_TO_DRIVE = False   # set True in Colab to copy results and checkpoints to Google Drive (survives the runtime)

HPARAMS = dict(
    total_timesteps=200_000,   # starting guess, not tuned
    learning_rate=3e-4,
    gamma=0.99,
    gae_lambda=0.95,
    num_steps=2048,
    num_minibatches=32,
    update_epochs=10,
    clip_coef=0.2,
    ent_coef=0.0,
    vf_coef=0.5,
    hidden=(64, 64),
)

random.seed(EXPERIMENT["train_seed"])
np.random.seed(EXPERIMENT["train_seed"])
torch.manual_seed(EXPERIMENT["train_seed"])
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

**2.3 Shared, identical in all three notebooks. Do not edit.** Results file schema (so one comparison notebook can read all three), checkpoint save and load (the viewer loads these), Drive backup, and the learning-curve plot.

In [ ]:
import json, datetime, subprocess, shutil
import matplotlib.pyplot as plt
try:
    from IPython.display import display
except ImportError:
    display = print

RESULTS_DIR = os.path.join(REPO, "results")
CHECKPOINT_DIR = os.path.join(REPO, "checkpoints")

def git_commit():
    try:
        return subprocess.check_output(["git", "rev-parse", "--short", "HEAD"], cwd=REPO, text=True).strip()
    except Exception:
        return None

def new_results():
    return dict(
        schema=1,
        algorithm=EXPERIMENT["algorithm"],
        student=dict(name=EXPERIMENT["student_name"], id=EXPERIMENT["student_id"]),
        env_id=EXPERIMENT["env_id"],
        git_commit=git_commit(),
        created=datetime.datetime.now().isoformat(timespec="seconds"),
        hparams=HPARAMS,
        training=dict(train_seed=EXPERIMENT["train_seed"], total_timesteps=None, wall_clock_s=None,
                      best_checkpoint=None, curve=[]),   # curve: list of {"timesteps": int, "mean_return": float}
        evaluation=dict(eval_seeds=EXPERIMENT["eval_seeds"], baselines={}, levels={}),
    )

def save_results(results):
    os.makedirs(RESULTS_DIR, exist_ok=True)
    path = os.path.join(RESULTS_DIR, f"{EXPERIMENT['algorithm'].lower()}_{EXPERIMENT['student_id']}.json")
    with open(path, "w") as f:
        json.dump(results, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))
    return path

def save_checkpoint(model, obs_dim, act_dim, tag="best"):
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    path = os.path.join(CHECKPOINT_DIR, f"{EXPERIMENT['algorithm'].lower()}_{EXPERIMENT['student_id']}_{tag}.pt")
    torch.save(dict(algorithm=EXPERIMENT["algorithm"], student_id=EXPERIMENT["student_id"], hparams=HPARAMS,
                    obs_dim=obs_dim, act_dim=act_dim, state_dict=model.state_dict()), path)
    return path

def load_checkpoint(model, tag="best"):
    path = os.path.join(CHECKPOINT_DIR, f"{EXPERIMENT['algorithm'].lower()}_{EXPERIMENT['student_id']}_{tag}.pt")
    ckpt = torch.load(path, map_location="cpu")
    model.load_state_dict(ckpt["state_dict"])
    return ckpt

def backup_to_drive(dirs):
    if not IN_COLAB:
        print("Not in Colab: nothing to back up.")
        return
    from google.colab import drive
    drive.mount("/content/drive")
    dest = "/content/drive/MyDrive/agroflight"
    os.makedirs(dest, exist_ok=True)
    for d in dirs:
        if os.path.isdir(d):
            shutil.copytree(d, os.path.join(dest, os.path.basename(d)), dirs_exist_ok=True)
    print("backed up to", dest)

def plot_learning_curve(curve, window=10):
    xs = [p["timesteps"] for p in curve]
    ys = np.array([p["mean_return"] for p in curve], dtype=float)
    fig, ax = plt.subplots(figsize=(8, 3.6))
    ax.plot(xs, ys, alpha=0.35, label="mean return")
    if len(ys) >= window:
        smooth = np.convolve(ys, np.ones(window) / window, mode="valid")
        ax.plot(xs[window - 1:], smooth, linewidth=2, label=f"moving average ({window})")
    ax.set_xlabel("environment steps")
    ax.set_ylabel("episode return")
    ax.legend()
    ax.grid(alpha=0.25)
    plt.show()

RESULTS = new_results()

**2.4 Environment check.** One reset and one random step, to confirm the model loads.

In [ ]:
import gymnasium as gym
import agroflight.envs
from agroflight.envs import ENV_ID

env = gym.make(ENV_ID, level=EXPERIMENT["level"])
obs, _ = env.reset(seed=EXPERIMENT["train_seed"])
print("observation:", env.observation_space.shape, "| action:", env.action_space.shape)
obs, r, terminated, truncated, info = env.step(env.action_space.sample())
print("one random step -> reward", round(r, 3), "| crashed:", info["crashed"])
env.close()

<a id="theory"></a>

## 3. Theory

<div style="border-left:4px solid rgb(147,105,28);background:rgba(147,105,28,0.10);padding:10px 14px;border-radius:0 8px 8px 0;margin:6px 0"><b>Write here.</b> Explain PPO in your own words, as if teaching a teammate. Cover:<ul><li>What the probability ratio $r_t(\theta)$ measures, and what the clip does to it</li><li>Why clipping approximates a trust region without second-order maths</li><li>Generalised Advantage Estimation (GAE): what $\lambda$ trades off</li><li>Why PPO reuses each batch for several epochs, and why that is safe here</li><li>Which implementation details CleanRL adds beyond the paper (for example advantage normalisation and learning-rate annealing)</li><li>Gaussian policy head for the continuous action space</li></ul>Add short pseudocode. The oral defence asks about exactly this section.</div>

Core objective, as a starting point to explain (not a substitute for the explanation):

$$L^{CLIP}(\theta)=\mathbb{E}_t\Big[\min\big(r_t(\theta)\hat A_t,\ \mathrm{clip}(r_t(\theta),1-\epsilon,1+\epsilon)\hat A_t\big)\Big],\qquad r_t(\theta)=\frac{\pi_\theta(a_t\mid s_t)}{\pi_{\theta_{old}}(a_t\mid s_t)}$$

<a id="implementation"></a>

## 4. Implementation

Adapt CleanRL's single-file `ppo_continuous_action.py` (to be copied into `src/agroflight/training/`, because the pip `cleanrl` 0.4.8 wrapper fails on import). Read it line by line and explain each block in your own words; the explanation is what gets marked at the oral defence.

In [ ]:
import torch
import torch.nn as nn
from torch.distributions import Normal

class Agent(nn.Module):
    """Actor-critic for PPO: obs -> action distribution and obs -> value."""
    def __init__(self, obs_dim, act_dim, hidden=HPARAMS["hidden"]):
        super().__init__()
        # TODO: critic network (obs -> scalar value)
        # TODO: actor mean network (obs -> action mean) and a learnable actor_logstd parameter

    def get_value(self, obs):
        raise NotImplementedError  # TODO

    def get_action_and_value(self, obs, action=None):
        raise NotImplementedError  # TODO: sample, log-prob, entropy, value

def compute_gae(rewards, values, dones, next_value, gamma, gae_lambda):
    raise NotImplementedError  # TODO: advantages and returns

def ppo_update(agent, optimizer, batch):
    raise NotImplementedError  # TODO: clipped surrogate + value loss + entropy bonus, several epochs of minibatches

<a id="training"></a>

## 5. Training

<div style="border-left:4px solid rgb(147,105,28);background:rgba(147,105,28,0.10);padding:10px 14px;border-radius:0 8px 8px 0;margin:6px 0"><b>Write here.</b> Describe the training setup: the difficulty level (start at L0 and only move up when it is solved), hyperparameters and why, wall-clock time per 1M steps (Part B asks Stage 1 to calibrate this), and at least 3 seeds for the stability comparison.</div>

In [ ]:
# TODO: training loop for PPO.
# After every N steps append a point to the curve, for example:
# RESULTS["training"]["curve"].append({"timesteps": global_step, "mean_return": mean_episode_return})
# When finished, set RESULTS["training"]["total_timesteps"] and RESULTS["training"]["wall_clock_s"].
#
# Keep the best model by TRAINING-time return, never by the held-out evaluation seeds:
# RESULTS["training"]["best_checkpoint"] = save_checkpoint(model, obs_dim=14, act_dim=4, tag="best")

In [ ]:
if RESULTS["training"]["curve"]:
    plot_learning_curve(RESULTS["training"]["curve"])
else:
    print("No training run recorded yet.")

<a id="evaluation"></a>

## 6. Evaluation

Every policy is evaluated on the same held-out seeds (1000 to 1019) with exploration disabled. The two hand-coded baselines run at this notebook's level (`EXPERIMENT["level"]`, L0 by default); re-run them if the environment changes.

In [ ]:
from agroflight.evaluation.evaluate_policy import evaluate
from agroflight.baselines import altitude_hold_action, random_action

eval_env = gym.make(ENV_ID, level=EXPERIMENT["level"])
rng = np.random.default_rng(0)

RESULTS["evaluation"]["baselines"]["altitude_hold"] = evaluate(
    altitude_hold_action, seeds=EXPERIMENT["eval_seeds"], env=eval_env)
RESULTS["evaluation"]["baselines"]["random"] = evaluate(
    lambda obs: random_action(eval_env.action_space, rng), seeds=EXPERIMENT["eval_seeds"], env=eval_env)

for name, st in RESULTS["evaluation"]["baselines"].items():
    print(f"{name:<14} reward {st['mean_reward']:8.1f}  crash {st['crash_rate']:.0%}  finish {st['finish_rate']:.0%}")

In [ ]:
# TODO: after training, rebuild the model, call load_checkpoint(model, tag="best"), and set trained_policy
# to a function obs -> action (deterministic, no exploration).
trained_policy = None

# TODO: evaluate at every difficulty level this algorithm reaches (L0 to L4), creating one env per level,
# and store one entry per level, for example RESULTS["evaluation"]["levels"]["L0"] = evaluate(...)
if trained_policy is not None:
    RESULTS["evaluation"]["levels"]["L0"] = evaluate(trained_policy, seeds=EXPERIMENT["eval_seeds"], env=eval_env)
else:
    print("No trained PPO policy yet: baselines only.")

<a id="results"></a>

## 7. Results

In [ ]:
import pandas as pd

rows = []
for name, st in RESULTS["evaluation"]["baselines"].items():
    rows.append({"policy": name, **st})
for level, st in RESULTS["evaluation"]["levels"].items():
    rows.append({"policy": f"PPO @ {level}", **st})

cols = ["mean_reward", "std_reward", "crash_rate", "finish_rate", "mean_steps", "mean_abs_alt_error", "n_episodes"]
df = pd.DataFrame(rows).set_index("policy")[cols]
display(df.round(3))

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(11, 3.2))
for ax, col, title in zip(axes, ["mean_reward", "crash_rate", "finish_rate"],
                          ["Mean reward", "Crash rate", "Row completion rate"]):
    ax.bar(df.index, df[col])
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=25)
    ax.grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()

path = save_results(RESULTS)
print("results saved to", path)
if BACKUP_TO_DRIVE:
    backup_to_drive([RESULTS_DIR, CHECKPOINT_DIR])

<div style="border-left:4px solid rgb(147,105,28);background:rgba(147,105,28,0.10);padding:10px 14px;border-radius:0 8px 8px 0;margin:6px 0"><b>Write here.</b> State the headline result in one sentence: the highest level cleared and the samples it needed. Compare against both baselines and against your success criterion from Section 1. Report what did not work as well as what did.</div>

<a id="discussion"></a>

## 8. Discussion

<div style="border-left:4px solid rgb(147,105,28);background:rgba(147,105,28,0.10);padding:10px 14px;border-radius:0 8px 8px 0;margin:6px 0"><b>Write here.</b> <ul><li>Was training stable across seeds? Show the spread, not just the best run.</li><li>Did the reward behave as intended, or did you see reward hacking (for example hovering in place)? Check the viewer, not only the number.</li><li>Where did the algorithm fail, and is that the algorithm, the reward, or the environment?</li><li>Limits: MuJoCo has no rotor ground effect or canopy turbulence; wind is observable; single environment.</li><li>What would you try next with more time?</li></ul></div>

<a id="contribution"></a>

## 9. Contribution and reproducibility

<div style="border-left:4px solid rgb(147,105,28);background:rgba(147,105,28,0.10);padding:10px 14px;border-radius:0 8px 8px 0;margin:6px 0"><b>Write here.</b> Contribution statement (marks are split by individual contribution). Fill in the table.</div>

| | |
|---|---|
| **My contribution** | *what Nelkit built, ran and wrote* |
| **Shared work with teammates** | *environment rework, evaluation harness, viewer, report sections* |
| **AI assistance used** | *which tool, for what, and what you verified yourself* |

In [ ]:
import platform
print("python    :", platform.python_version())
print("torch     :", torch.__version__)
print("numpy     :", np.__version__)
print("gymnasium :", gym.__version__)
print("mujoco    :", mujoco.__version__)
print("git commit:", git_commit())
print("train seed:", EXPERIMENT["train_seed"], "| eval seeds:", EXPERIMENT["eval_seeds"][0], "to", EXPERIMENT["eval_seeds"][-1])
print("notebook wall-clock so far: %.0f s" % (time.time() - NOTEBOOK_T0))

### References

1. Schulman, J., Wolski, F., Dhariwal, P., Radford, A., Klimov, O. (2017). Proximal Policy Optimization Algorithms. arXiv:1707.06347.
2. Schulman, J., Moritz, P., Levine, S., Jordan, M., Abbeel, P. (2016). High-Dimensional Continuous Control Using Generalized Advantage Estimation. ICLR. arXiv:1506.02438.
3. Huang, S. et al. (2022). CleanRL: High-quality Single-file Implementations of Deep Reinforcement Learning Algorithms. JMLR 23(274).
4. Todorov, E., Erez, T., Tassa, Y. (2012). MuJoCo: A physics engine for model-based control. IROS.
5. Google DeepMind. MuJoCo Menagerie: Skydio X2 model (Apache-2.0). github.com/google-deepmind/mujoco_menagerie.
6. Towers, M. et al. (2024). Gymnasium: A Standard Interface for Reinforcement Learning Environments. arXiv:2407.17032.